# Walk-forward threshold search: clay & hard/grass

Reproducible pipeline for:
1. Loading `CRITERIA_DATA.xlsx` (excluding `US_hard`, `WE_hard` — duplicated by seasonal splits)
2. Reconstructing `win2`/`win1` flat-stake profit columns where missing
3. Searching the k2 threshold that maximizes cumulative win2 (min 300 matches), per group per window
4. Building walk-forward (no time-leak) `prev.seasons` / `curr.season` tables per `tourn_sheet`

See `data/processed/config.yaml` for the full methodology spec.

## Setup

In [ ]:
import pandas as pd

DATA_FILE = '../data/raw/CRITERIA_DATA.xlsx'  # adjust path as needed

def compute_win2(row):
    if row['win'] == 0:
        return 100.0
    return -100.0 / (row['k2'] - 1) if row['k2'] != 1 else None

def compute_win1(row):
    if row['win'] == 1:
        return 100.0
    return -100.0 / (row['k1'] - 1) if row['k1'] != 1 else None

def load_sheet(sheet_name):
    df = pd.read_excel(DATA_FILE, sheet_name=sheet_name)
    df['__sheet'] = sheet_name
    if 'win2' not in df.columns:
        df['win2'] = df.apply(compute_win2, axis=1)
    if 'win1' not in df.columns:
        df['win1'] = df.apply(compute_win1, axis=1)
    df['year'] = pd.to_datetime(df['Date']).dt.year
    return df[['Date','Loc','Rd','PL1','PL2','k1','k2','win','win1','win2','__sheet','year']]

## Threshold search function

Sorts by ascending k2, aggregates by unique k2 (avoids order-dependence on ties),
computes cumulative sum of win2, returns the k2 with max cumulative sum subject
to a minimum match-count constraint.

In [ ]:
def find_threshold(df, min_count=300):
    agg = df.groupby('k2')['win2'].agg(['sum','count']).reset_index().sort_values('k2')
    agg['cum_win2'] = agg['sum'].cumsum()
    agg['cum_count'] = agg['count'].cumsum()
    valid = agg[agg['cum_count'] >= min_count]
    if valid.empty:
        return None, None, None
    best = valid.loc[valid['cum_win2'].idxmax()]
    return best['k2'], best['cum_win2'], int(best['cum_count'])

WINDOWS = {
    'до 2020 вкл': (None, '2020-12-31 23:59:59'),
    'до 2021 вкл': (None, '2021-12-31 23:59:59'),
    '2016-2022 вкл': ('2016-01-01', '2022-12-31 23:59:59'),
    '2017-2023 вкл': ('2017-01-01', '2023-12-31 23:59:59'),
    '2018-2024 вкл': ('2018-01-01', '2024-12-31 23:59:59'),
    '2019-2025 вкл': ('2019-01-01', '2025-12-31 23:59:59'),
}
WINDOW_END_YEAR = {
    'до 2020 вкл': 2020, 'до 2021 вкл': 2021, '2016-2022 вкл': 2022,
    '2017-2023 вкл': 2023, '2018-2024 вкл': 2024, '2019-2025 вкл': 2025,
}

def thresholds_for_groups(group_dfs, min_count=300):
    rows = []
    for label, (start, end) in WINDOWS.items():
        for gname, gdf in group_dfs.items():
            d = gdf.copy()
            if start: d = d[d['Date'] >= start]
            if end: d = d[d['Date'] <= end]
            k2_thr, cum_max, cnt = find_threshold(d, min_count)
            rows.append([label, WINDOW_END_YEAR[label], gname, k2_thr, cum_max, cnt, len(d)])
    return pd.DataFrame(rows, columns=['Window','window_end_year','Group',
                                         'Threshold_k2','Max_cum_win2',
                                         'Matches_under_threshold','Total_matches'])

def no_leak_year_thresholds(thresholds_df):
    """Shift each window's threshold forward by 1 year: threshold computed on
    a window ending at Y-1 is used to evaluate year Y (prevents time leak)."""
    out = {}
    for _, row in thresholds_df.iterrows():
        target_year = row['window_end_year'] + 1
        if target_year <= 2025:
            out[(row['Group'], target_year)] = row['Threshold_k2']
    return out

def results_by_sheet(sheet_to_group, sheet_dfs, threshold_by_year_group, years=range(2021,2026)):
    rows = []
    for sheet, gname in sheet_to_group.items():
        sdf = sheet_dfs[sheet]
        for year in years:
            threshold = threshold_by_year_group[(gname, year)]
            prev_years = [year-3, year-2, year-1]
            prev_df = sdf[(sdf['year'].isin(prev_years)) & (sdf['k2'] <= threshold)]
            curr_df = sdf[(sdf['year'] == year) & (sdf['k2'] <= threshold)]
            rows.append({
                'tourn_sheet': sheet, 'group': gname,
                'prev.seasons': prev_df['win2'].sum(), 'prev_count': len(prev_df),
                'curr.season': curr_df['win2'].sum(), 'curr_count': len(curr_df),
                'year': year, 'threshold': threshold,
            })
    return pd.DataFrame(rows)

## Clay: SAm_Eu_Clay vs Masters_RG (5 sheets, no RG_Q/RG split)

In [ ]:
CLAY_GROUPS = {
    'SAm_Eu_Clay': ['SAm_clay', 'Eu_Clay'],
    'Masters_RG': ['Madrid', 'Monte-Carlo', 'Rome', 'RG_Q', 'RG'],
}
clay_sheet_to_group = {s: g for g, sheets in CLAY_GROUPS.items() for s in sheets}
clay_sheet_dfs = {s: load_sheet(s) for s in clay_sheet_to_group}
clay_group_dfs = {g: pd.concat([clay_sheet_dfs[s] for s in sheets], ignore_index=True)
                  for g, sheets in CLAY_GROUPS.items()}

clay_thresholds = thresholds_for_groups(clay_group_dfs, min_count=300)
clay_year_thr = no_leak_year_thresholds(clay_thresholds)
clay_results = results_by_sheet(clay_sheet_to_group, clay_sheet_dfs, clay_year_thr)

clay_thresholds_export = clay_thresholds[['Window','window_end_year','Group','Threshold_k2',
                                            'Max_cum_win2','Matches_under_threshold','Total_matches']]
clay_results_export = clay_results[['tourn_sheet','group','prev.seasons','curr.season',
                                      'year','threshold','prev_count','curr_count']]

with pd.ExcelWriter('../data/processed/clay_thresholds_and_results.xlsx', engine='openpyxl') as writer:
    clay_thresholds_export.to_excel(writer, sheet_name='Thresholds', index=False)
    clay_results_export.to_excel(writer, sheet_name='Results_by_tourn_sheet', index=False)

## Hard/Grass: ATP_hard_Masters, GrandSlam_hard_grass, ATP_grass (17 sheets)

In [ ]:
HARD_GRASS_GROUPS = {
    'ATP_hard_Masters': ['WE_hard_feb','WE_hard_oct','US_hard_feb','US_hard_summer','Aus','Chn',
                          'US1000_march','US1000_aug','Chn_masters','Paris'],
    'GrandSlam_hard_grass': ['AO','AO_Q','US_open','US_Q','Wimbledon','Wim_Q'],
    'ATP_grass': ['grass'],
}
hg_sheet_to_group = {s: g for g, sheets in HARD_GRASS_GROUPS.items() for s in sheets}
hg_sheet_dfs = {s: load_sheet(s) for s in hg_sheet_to_group}
hg_group_dfs = {g: pd.concat([hg_sheet_dfs[s] for s in sheets], ignore_index=True)
                for g, sheets in HARD_GRASS_GROUPS.items()}

hg_thresholds = thresholds_for_groups(hg_group_dfs, min_count=300)
hg_year_thr = no_leak_year_thresholds(hg_thresholds)
hg_results = results_by_sheet(hg_sheet_to_group, hg_sheet_dfs, hg_year_thr)

hg_thresholds_export = hg_thresholds[['Window','window_end_year','Group','Threshold_k2',
                                        'Max_cum_win2','Matches_under_threshold','Total_matches']]
hg_results_export = hg_results[['tourn_sheet','group','prev.seasons','curr.season',
                                  'year','threshold','prev_count','curr_count']]

with pd.ExcelWriter('../data/processed/three_groups_hard_grass_thresholds.xlsx', engine='openpyxl') as writer:
    hg_thresholds_export.to_excel(writer, sheet_name='Thresholds', index=False)
    hg_results_export.to_excel(writer, sheet_name='Results_by_tourn_sheet', index=False)

## Notes

- **No time leak**: threshold for evaluating year Y always comes from a window ending Y-1.
- **win2 formula validated**: on sheets with a native `win2` column (RG, SAm_clay, Eu_Clay, US_hard),
  the reconstructed formula matches exactly (max abs diff = 0).
- **US_hard / WE_hard excluded**: fully covered by `US_hard_feb`+`US_hard_summer` and
  `WE_hard_feb`+`WE_hard_oct` respectively (verified by exact key-set equality).